# Lecture 6b: THE AGENT'S SIDE - Verifying Inclusion (one of many)

> **DOMAIN BANNER - read this first.** In this notebook YOU ARE AN
> AGENT. There are **many** of you. You possess exactly three
> things: the provider's public key, the evidence files, and about
> forty lines of hashing code. You do NOT possess Lean, a proof
> toolchain, or the provider's private key - and you never will
> need them. Everything below runs in milliseconds. If a cell in
> this notebook needed Lean, the design would have failed.

This is **Merkle proof verification, not Lean verification** - the
agent checks WHERE a statement sits, never re-derives WHY it is true.


## Learning Objectives

- Implement the complete inclusion verifier from scratch - hashlib only, no pacta imports for the core.
- Verify a REAL receipt against the REAL signed tree head.
- See the inclusion path in the picture of the real 8-leaf log.
- Read the provider's self-check ("dogfood in both directions") from the signature block and say what it does and does not prove.


## The whole verifier, from scratch

To make the cost asymmetry unmistakable, here is the ENTIRE core of
what an agent must implement - RFC 9162 inclusion verification in
~25 lines of standard-library Python. Read every line; this is all
the cryptographic machinery your trust rests on (plus one Ed25519
signature check):


In [ ]:
import hashlib

def leaf_hash(data: bytes) -> bytes:
    return hashlib.sha256(b"\x00" + data).digest()

def node_hash(left: bytes, right: bytes) -> bytes:
    return hashlib.sha256(b"\x01" + left + right).digest()

def verify_inclusion(leaf: bytes, index: int, size: int, proof: list, root: bytes) -> bool:
    if index >= size:
        return False
    fn, sn = index, size - 1
    node = leaf_hash(leaf)
    for sibling in proof:
        if sn == 0:
            return False
        if fn % 2 == 1 or fn == sn:
            node = node_hash(sibling, node)
            if fn % 2 == 0:
                while fn % 2 == 0 and fn != 0:
                    fn //= 2
                    sn //= 2
        else:
            node = node_hash(node, sibling)
        fn //= 2
        sn //= 2
    return sn == 0 and node == root

print("the agent's entire Merkle toolbox: 3 functions,", "no imports beyond hashlib")


## Apply it to the REAL receipt


In [ ]:
import json
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "src" / "pacta").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))
from pacta.yamlio import load_data
from pacta.signing import canonical_json

att = load_data(repo_root / "evidence" / "dalek-ed25519.attestation.yaml")
receipt = load_data(repo_root / "evidence" / "dalek-ed25519.receipt.yaml")

leaf_bytes = canonical_json({"schema_version": 1, "type": "pacta.transparency.attestation_leaf.v1", "attestation": att})
proof = [bytes.fromhex(h) for h in receipt["inclusion_proof"]]
root = bytes.fromhex(receipt["sth"]["root_hash"])

ok = verify_inclusion(leaf_bytes, receipt["leaf_index"], receipt["tree_size"], proof, root)
print(f"leaf {receipt['leaf_index']} of {receipt['tree_size']}, {len(proof)} siblings -> inclusion:", ok)
assert ok


## One signature check completes the chain

Inclusion binds the attestation to a root; the signature binds the
root to the provider. Note what the AGENT learns from the signature
block's `signing_provenance`: the provider signed this root with the
merkleized library and Merkle-verified that library's own leaf first
- dogfood in both directions. The agent still re-checks inclusion
itself (above); the provenance is the provider's discipline made
visible, not a substitute for the agent's check.


In [ ]:
from pacta.transparency import verify_signed_tree_head
from pacta.sthstore import check_sth_against_store
import tempfile

ok, diagnostics, statuses = verify_signed_tree_head(receipt["sth"], repo_root / "evidence" / "provider.ed25519.pub")
print("STH signature:", statuses.get("ed25519"), "| verified on backend:", statuses.get("ed25519_backend"))
print("provider's own discipline, as recorded in the signature block:")
print(json.dumps(receipt["sth"]["signatures"]["ed25519"].get("signing_provenance", {}), indent=1))
with tempfile.TemporaryDirectory() as tmp:
    pin = check_sth_against_store(receipt["sth"], Path(tmp) / "pins.json")
    print("pin store:", pin.action)


## The picture: where your leaf sits in the real log


In [ ]:
# reconstruct the on-path node hashes from the receipt alone - the
# agent never needs the other leaves, only the siblings.
def svg_inclusion(receipt, width=980):
    size, index = receipt["tree_size"], receipt["leaf_index"]
    proof = receipt["inclusion_proof"]
    rows = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="330" font-family="monospace" font-size="11">']
    rows.append(f'<rect x="4" y="4" width="{width-8}" height="322" fill="none" stroke="#3b4d8f" stroke-width="2" rx="8"/>')
    rows.append('<text x="16" y="24" fill="#3b4d8f" font-size="13" font-weight="bold">AGENT: verifies only the highlighted path - everything grey is somebody else&apos;s data</text>')
    span = width / (size + 1)
    for i in range(size):
        x = span * (i + 1)
        me = i == index
        fill = "#e2f2e9" if me else "#f4f4f6"
        stroke = "#1e7f4f" if me else "#bbb"
        rows.append(f'<rect x="{x-42}" y="250" width="84" height="30" fill="{fill}" stroke="{stroke}" stroke-width="{2 if me else 1}" rx="4"/>')
        label = "YOUR leaf" if me else f"leaf {i}"
        rows.append(f'<text x="{x}" y="269" text-anchor="middle" fill="{stroke}">{label}</text>')
    node = leaf_hash(leaf_bytes)
    y = 250
    x = span * (index + 1)
    for depth, sib in enumerate(proof):
        ny = y - 60
        nx = x  # visual simplification: path rises vertically
        rows.append(f'<rect x="{nx-52}" y="{ny-16}" width="104" height="30" fill="#fdf0da" stroke="#a86a10" rx="4"/>')
        rows.append(f'<text x="{nx}" y="{ny+3}" text-anchor="middle" fill="#a86a10">sibling {depth}: {sib[:10]}…</text>')
        rows.append(f'<line x1="{x}" y1="{y-16 if depth else 250}" x2="{nx}" y2="{ny+14}" stroke="#a86a10"/>')
        y = ny
    rows.append(f'<rect x="{x-135}" y="{y-70}" width="270" height="30" fill="#eef" stroke="#3b4d8f" rx="4"/>')
    rows.append(f'<text x="{x}" y="{y-50}" text-anchor="middle" fill="#3b4d8f">signed root {receipt["sth"]["root_hash"][:14]}… (dogfood-signed)</text>')
    rows.append(f'<line x1="{x}" y1="{y-16}" x2="{x}" y2="{y-40}" stroke="#3b4d8f"/>')
    rows.append("</svg>")
    return "".join(rows)

svg = svg_inclusion(receipt)
try:
    from IPython.display import SVG, display
    display(SVG(svg))
except Exception:
    print("(open in a notebook to render the figure)")
print(f"cost of everything in this notebook: ~{receipt['tree_size'].bit_length()} hashes + 1 signature check - milliseconds.")


## Convinced - of what, exactly?

After these cells pass, the agent knows: *the provider whose key I
pinned states that the Lean proofs of repository X at commit Y check
out with exactly the documented assumptions, and that statement is
irrevocably part of the log every other agent sees.* The agent then
clones commit Y (the git hash IS the content hash) and builds it -
compiler and build remain declared trusted base until R5. Where a
claim lives (this notebook) and why it is true (the provider's Lean
replay, lecture 6a) never blur.

## Exercises

- Flip one byte of the leaf and re-run: which of the ~25 lines catches it?
- Flip one byte of the ROOT instead: what fails now, inclusion or the signature?
- Your entire verifier fits in one cell. List everything it does NOT check (freshness? consistency with your previous pin? provider honesty about Lean?) and name the lecture that closes each gap.
- The provenance block says the provider checked itself. Why must the agent still run its own inclusion check rather than trust that field?
